In [2]:
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_chroma import Chroma

c:\Users\Asus\source\repos\prakash-manit\AgenticAI\Python\VectorDb\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader = WebBaseLoader(web_path="https://techyprakash.com/about/")
docs = loader.load()
print(f"Number of documents loaded: {len(docs)}")
print(docs)
print(f"Total characters: {len(docs[0].page_content)}")

Number of documents loaded: 1
[Document(metadata={'source': 'https://techyprakash.com/about/', 'title': 'About – Techy Prakash', 'description': 'Purpose: Purpose of this site is to promote efficient coding and future-ready design practices by creating useful content around computer science fundamentals and modern tech such as Data Structure & Algorithms, System Design, Cloud, AI, Best practices etc. Blog Owner: This blog is maintained by Prakash Tripathi who is an Engineering Leader by profession (17+…', 'language': 'en'}, page_content='\n\n\n\n\n\n\n\nAbout – Techy Prakash\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLinkedIn\nYouTube\nTwitter\nFacebook\nGitHub\n\n\n\n\n\n\nHomeAbout\n\n\n\n\n\n\n\n\nAbout\n\nPurpose:\nPurpose of this site is to promote efficient coding and future-ready design practices by creating useful content around computer science fundamentals and modern tech such as Data S

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
print(all_splits)
print(f"Split blog post into: {len(all_splits)} sub-documents")

[Document(metadata={'source': 'https://techyprakash.com/about/', 'title': 'About – Techy Prakash', 'description': 'Purpose: Purpose of this site is to promote efficient coding and future-ready design practices by creating useful content around computer science fundamentals and modern tech such as Data Structure & Algorithms, System Design, Cloud, AI, Best practices etc. Blog Owner: This blog is maintained by Prakash Tripathi who is an Engineering Leader by profession (17+…', 'language': 'en'}, page_content='About – Techy Prakash\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLinkedIn\nYouTube\nTwitter\nFacebook\nGitHub\n\n\n\n\n\n\nHomeAbout\n\n\n\n\n\n\n\n\nAbout'), Document(metadata={'source': 'https://techyprakash.com/about/', 'title': 'About – Techy Prakash', 'description': 'Purpose: Purpose of this site is to promote efficient coding and future-ready design practices by creating useful content 

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = Chroma(collection_name="techyprakash_info", embedding_function=embeddings, persist_directory="./chroma_genai")

C:\Users\Asus\AppData\Local\Temp\ipykernel_9896\570044085.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(collection_name="techyprakash_info", embedding_function=embeddings, persist_directory="./Chroma_genai")


In [6]:
vectorstore.add_documents(documents=all_splits)
print(vectorstore._collection.count())  # Check total stored chunks

4


In [ ]:
@tool
def retrieve_context(query: str):
  """Search for info related to Techy Prakash"""
  try:
      embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
      for i, doc in enumerate(all_splits, start=1):
           vec = embeddings.embed_query(doc.page_content)  # embedding for this chunk
           print(f"\n--- Chunk {i} ---")
           print("Text snippet:", doc.page_content[:200], "...")
           print("Embedding length:", len(vec))
           print("Embedding preview (first 5 dims):", vec[:5])
           vector_store = Chroma(collection_name="techyprakash_info", embedding_function=embeddings, persist_directory="./chroma_genai")
      
      retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

      print(f"Querying retrieve_context with: {query}")
      print("--------------------------------------------------------------")
      results = retriever.invoke(query)
      print(f"Retrieved documents: {len(results)} matches found")
      for i, doc in enumerate(results):
          print(f"Document {i + 1}: {doc.page_content[:100]}...")
    
      print("--------------------------------------------------------------")

      content = "\n".join([doc.page_content for doc in results])
      if not content:
          print(f"No content retrieved for query: {query}")
          return f"No reviews found for '{query}'."
    
      print("--------------------------------------------------------------")
      print(f"Returning content: {content[:200]}...")
      return content
  except Exception as e:
      print(f"Error in retrieve_context: {e}")
      return f"Error retrieving reviews for '{query}'. Please try again."

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [9]:
agent_executor = create_react_agent(llm, [retrieve_context])

C:\Users\Asus\AppData\Local\Temp\ipykernel_9896\4122366744.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, [retrieve_context])


In [12]:
input_message = (
  "give me name of the person who is blogging in techy prakash?"
)
for event in agent_executor.stream(
  {"messages": [{"role": "user", "content": input_message}]},
  stream_mode="values"
):
  event["messages"][-1].pretty_print()

================================ Human Message =================================

give me name of the person who is blogging in techy prakash?


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 8.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 16.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


KeyboardInterrupt: 